# 02. 크롤링 파이프라인 — Logic App 수동 실행 및 검증

이 노트북의 목적은 **Logic App 크롤 파이프라인이 end-to-end 로 잘 도는지** 확인하는 것입니다. 개별 Function 호출이 아닌 Logic App orchestration 흐름을 검증합니다.

1. **환경 설정** — `.env` 변수 로드
2. **Blob Storage 현황** — 이미 크롤링된 데이터가 있는지 확인
3. **Logic App 상태 확인 및 수동 실행** — workflow 상태 점검 후 `Daily_Schedule_Crawl_and_Preprocess` 트리거 즉시 호출
4. **실행 이력 확인** — 최근 run 의 성공/실패 모니터링
5. **AI Search 인덱서 상태 확인** — 색인 결과 검증

> 파이프라인 흐름: Logic App → Durable Functions HTTP Starter (`crawl_preprocess`) → 4 source × wave fan-out (listing → detail → preprocess) → Blob 업로드 → AI Search indexer 자동 트리거


## 1. 환경 설정

In [1]:
import os
import sys
import json
import subprocess
from datetime import datetime
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

CRAWL_URL    = os.environ.get('AZURE_FUNCTION_CRAWL_URL', '')
STORAGE_NAME = os.environ.get('AZURE_STORAGE_ACCOUNT_NAME', '')
CONTAINER    = os.environ.get('AZURE_STORAGE_CONTAINER_NAME', 'raw-documents')
RG_NAME      = os.environ.get('AZURE_RESOURCE_GROUP', 'rg-rag-indexing-lab-swc')
LOGIC_APP    = os.environ.get('AZURE_LOGIC_APP_NAME', '')

print(f"Function URL : {CRAWL_URL}")
print(f"Storage      : {STORAGE_NAME}")
print(f"Container    : {CONTAINER}")
print(f"Logic App    : {LOGIC_APP}")

Function URL : https://func-crawl-cons-ragi-63325wdo.azurewebsites.net/api/orchestrators/crawl_preprocess
Storage      : stragi63325wdoby
Container    : raw-documents
Logic App    : logic-crawl-ragi-63325wdo


## 2. Blob Storage 크롤링 결과 확인

현재 저장된 크롤링 데이터의 파일 개수를 확인합니다.

In [3]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

# MSI_ENDPOINT (VS Code/Azure ML compute) 가 설정된 환경에서는 MI 토큰이
# az CLI 와 다른 테넌트로 발급돼 'Issuer validation failed' 가 발생할 수 있음.
# → MI/Workload Identity 를 제외하고 az CLI / 개발자 자격증명만 사용.
credential = DefaultAzureCredential(
    exclude_managed_identity_credential=True,
    exclude_workload_identity_credential=True,
)
blob_svc = BlobServiceClient(
    account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
    credential=credential,
)
container_client = blob_svc.get_container_client(CONTAINER)

blobs = list(container_client.list_blobs())

# Blob 경로: {source}/{date}/{source}_{seq}.json
sources = sorted(set(b.name.split('/')[0] for b in blobs if '/' in b.name))

print(f"\n{'='*60}")
print(f"크롤링 데이터 현황")
print(f"{'='*60}")
print(f"총 {len(blobs):,}개 파일 | {len(sources)}개 소스 폴더\n")

if len(blobs) == 0:
    print("⚠️  아직 크롤링된 데이터가 없습니다.")
    
else:
    for src in sources:
        src_blobs = [b for b in blobs if b.name.startswith(f"{src}/")]
        dates = sorted(set(b.name.split('/')[1] for b in src_blobs if b.name.count('/') >= 2), reverse=True)
        print(f"  /{src}/")
        print(f"    전체: {len(src_blobs)}개 파일, {len(dates)}개 날짜")
        for d in dates[:3]:
            cnt = sum(1 for b in src_blobs if f"{src}/{d}/" in b.name)
            print(f"      {d}: {cnt}개")
        print()


크롤링 데이터 현황
총 68,290개 파일 | 5개 소스 폴더

  /_logs/
    전체: 2471개 파일, 1개 날짜
      2026-05-22: 2471개

  /admrul/
    전체: 90개 파일, 1개 날짜
      2026-05-22: 90개

  /detc/
    전체: 9714개 파일, 1개 날짜
      2026-05-22: 9714개

  /prec/
    전체: 56013개 파일, 1개 날짜
      2026-05-22: 56013개

  /raw/
    전체: 2개 파일, 1개 날짜
      pdf: 2개



In [4]:
# 첫 번째 데이터 소스의 최신 JSON 파일 미리보기 (_logs 등 메타 폴더 제외)
data_sources = [s for s in sources if not s.startswith('_')]
if data_sources:
    for src in data_sources:
        
        json_blobs = sorted(
            [b for b in blobs if b.name.startswith(f"{src}/") and b.name.endswith('.json')],
            key=lambda b: b.name,
            reverse=True,
        )
        if json_blobs:
            blob = container_client.get_blob_client(json_blobs[0].name)
            content = blob.download_blob().readall().decode('utf-8')
            doc = json.loads(content)
            print(f"\n[샘플] {json_blobs[0].name}\n")
            pretty = json.dumps(doc, ensure_ascii=False, indent=2)
            print(pretty)
            # print(pretty[:1500])
            # if len(pretty) > 1500:
            #     print("\n... (생략)")
        else:
            print(f"\n/{src}/ 폴더에 .json 파일 없음")
else:
    print("\n데이터 소스 폴더가 없습니다.")


[샘플] admrul/2026-05-22/admrul_272881.json

{
  "seq": "272881",
  "source": "행정심판재결례",
  "사건명": "",
  "사건번호": "",
  "재결일자": "",
  "재결기관": "",
  "재결결과": "",
  "재결요지": "",
  "주문": "",
  "청구취지": "",
  "이유": "",
  "url": "https://www.law.go.kr/deccInfoP.do?deccSeq=272881"
}

[샘플] detc/2026-05-22/detc_9992.json

{
  "seq": "9992",
  "source": "헌재결정례",
  "사건명": "",
  "재판부": "",
  "사건번호": "",
  "결정일자": "",
  "판시사항": "",
  "결정요지": "",
  "심판대상조문": "",
  "참조조문": "",
  "참조판례": "",
  "전문": "",
  "url": "https://www.law.go.kr/detcInfoP.do?detcSeq=9992"
}

[샘플] prec/2026-05-22/prec_99997.json

{
  "seq": "99997",
  "source": "판례",
  "사건명": "관세환급금가산금지불의권리관계확인",
  "법원명": "대법원",
  "선고일자": "1984-12-26T00:00:00Z",
  "사건번호": "82누344",
  "판시사항": "관세추징금을 환급받음에 있어 납부액 이외에 가산금을 지급받을 권리가 있다는 권리관계의 확인을 구하는 것이 행정소송 사항인지 여부 (소극)",
  "판결요지": "관세추징부과처분취소 소송에서 승소한 자들이 위 부과처분을 한 행정청을 상대로 기납부한 추징금액 이외에 위 납부금액에 국세기본법 제52조 , 같은법시행령 제30조 의 규정을 유추적용한 가산금을 첨가하여 지급할 의무가 있다는 권리관계의 확인을 구하는 것은 민사소송으로 제기되어야 할 것이고 행정소송중 이른바 당사자

## 3. Durable Functions orchestration 직접 호출 (노트북 테스트 경로)

운영 환경에서는 Logic App `Daily_Schedule_Crawl_and_Preprocess` 가 매일 21:00 UTC (06:00 KST) 에 동일한 Durable orchestrator (`crawl_preprocess`) 를 호출합니다. 노트북에서는 Logic App 트리거를 거치지 않고 **orchestrator HTTP starter 를 직접 호출** 하여 결과를 폴링·검증합니다.

- 호출: `POST /api/orchestrators/crawl_preprocess` → 202 + `statusQueryGetUri` 즉시 반환
- 폴링: `statusQueryGetUri` 를 GET → `runtimeStatus` 가 `Completed` / `Failed` / `Terminated` 가 될 때까지 대기 (230s gateway 제한 없음)
- 출력: orchestrator 가 반환한 `sources/uploaded/docs` 합산

> 정기 운영용 Logic App workflow 는 그대로 유지됩니다. 아래 셀 4 에서 운영 모니터링(최근 run 이력) 확인.

In [5]:
import time
import requests

ORCH_URL = os.environ.get("AZURE_FUNCTION_CRAWL_ORCH_URL", "")
assert ORCH_URL, "AZURE_FUNCTION_CRAWL_ORCH_URL 가 .env 에 설정되어야 합니다."

# 노트북 검증용: 4개 소스 모두, 페이지당 100건 × 1페이지 만 (가볍게 end-to-end 확인)
orch_params = {
    "source": "all",
    "max_pages": "1",
    "triggered_by": "notebook-02",
}
print(f"▶ Durable orchestrator 호출: {ORCH_URL}")
print(f"  params={orch_params}")

# Function App 콜드 스타트 대응: 최대 3회, 각 120s timeout 으로 재시도
t0 = time.time()
start_resp = None
last_exc = None
for attempt in range(1, 4):
    try:
        start_resp = requests.post(ORCH_URL, params=orch_params, timeout=120)
        break
    except requests.exceptions.RequestException as e:
        last_exc = e
        print(f"  [attempt {attempt}] {type(e).__name__}: {e} — 10s 후 재시도")
        time.sleep(10)
if start_resp is None:
    raise RuntimeError(f"Orchestrator start 실패: {last_exc}")

print(f"  HTTP {start_resp.status_code}  (즉시 반환)")
if start_resp.status_code >= 400 or not start_resp.text.strip().startswith("{"):
    print(f"  ⚠️ 비정상 응답 (Content-Type: {start_resp.headers.get('Content-Type','')})")
    print(f"  Body: {start_resp.text[:500]}")
    raise RuntimeError(f"Orchestrator 응답이 JSON 아님 (HTTP {start_resp.status_code})")
start_json = start_resp.json()
instance_id  = start_json.get("id")
status_uri   = start_json["statusQueryGetUri"]
terminate_uri = start_json.get("terminatePostUri", "")
print(f"  instanceId    : {instance_id}")
print(f"  statusQueryUri: {status_uri[:120]}...")

# Polling
poll_interval = 15
max_wait = 1800  # 30분
last_status = ""
output = None
print(f"\n▶ 진행 상황 폴링 (interval={poll_interval}s, max_wait={max_wait}s)")
while True:
    elapsed = time.time() - t0
    if elapsed > max_wait:
        print(f"  ⚠️ max_wait {max_wait}s 초과 — 폴링 종료 (orchestration 은 계속 진행)")
        break
    try:
        r = requests.get(status_uri, timeout=30)
        pi = r.json()
    except Exception as e:
        print(f"  [{int(elapsed):4d}s] status 조회 실패: {e}")
        time.sleep(poll_interval)
        continue
    rt = pi.get("runtimeStatus", "?")
    if rt != last_status:
        print(f"  [{int(elapsed):4d}s] runtimeStatus = {rt}")
        last_status = rt
    if rt in ("Completed", "Failed", "Terminated"):
        output = pi.get("output")
        break
    time.sleep(poll_interval)

total_elapsed = time.time() - t0
print(f"\n▶ 종료 (total={total_elapsed:.0f}s)")

# Output 파싱: {status, crawl_date, triggered_by, crawl:{src:{...}}, preprocess:{src:{...}}}
if not isinstance(output, dict):
    print("⚠️ output 없음 또는 dict 아님:")
    pretty = json.dumps(pi, indent=2, ensure_ascii=False)
    print(pretty[:2000])
else:
    print(f"  status     : {output.get('status')}")
    print(f"  crawl_date : {output.get('crawl_date')}")
    print(f"  triggered_by: {output.get('triggered_by')}")
    crawl_map = output.get("crawl", {}) or {}
    pre_map = output.get("preprocess", {}) or {}
    sources = sorted(set(list(crawl_map.keys()) + list(pre_map.keys())))
    print(f"\n  ▼ 소스별 결과 ({len(sources)} sources)")
    header = f"  {'src':<8} {'crawl.saved':>12} {'crawl.skipExist':>16} {'pre.files':>10} {'pre.parts':>10} {'pre.docs':>10} {'pre.errors':>11}"
    print(header)
    print("  " + "-" * (len(header) - 2))
    g_saved = g_skip = g_files = g_parts = g_docs = g_errs = 0
    for src in sources:
        ci = crawl_map.get(src, {}) or {}
        pi = pre_map.get(src, {}) or {}
        saved = int(ci.get("saved_files", 0) or 0)
        skip  = int(ci.get("skipped_existing_listing", 0) or 0) + int(ci.get("skipped_duplicate", 0) or 0) + int(ci.get("pre_existing", 0) or 0)
        resp = (pi.get("response") or {}).get("sources", {}).get(src, {}) or {}
        files = int(resp.get("files", 0) or 0)
        parts = int(resp.get("parts", 0) or 0)
        errs  = int(resp.get("errors", 0) or 0)
        uploaded = resp.get("uploaded", []) or []
        docs  = sum(int((u or {}).get("docs", 0) or 0) for u in uploaded)
        g_saved += saved; g_skip += skip; g_files += files; g_parts += parts; g_docs += docs; g_errs += errs
        print(f"  {src:<8} {saved:>12} {skip:>16} {files:>10} {parts:>10} {docs:>10} {errs:>11}")
    print("  " + "-" * (len(header) - 2))
    print(f"  {'TOTAL':<8} {g_saved:>12} {g_skip:>16} {g_files:>10} {g_parts:>10} {g_docs:>10} {g_errs:>11}")


▶ Durable orchestrator 호출: https://func-crawl-cons-ragi-63325wdo.azurewebsites.net/api/orchestrators/crawl_preprocess
  params={'source': 'all', 'max_pages': '1', 'triggered_by': 'notebook-02'}


  HTTP 202  (즉시 반환)
  instanceId    : 4ccedddd001e46afb5b733314a246f13
  statusQueryUri: https://func-crawl-cons-ragi-63325wdo.azurewebsites.net/runtime/webhooks/durabletask/instances/4ccedddd001e46afb5b733314...

▶ 진행 상황 폴링 (interval=15s, max_wait=1800s)
  [   4s] runtimeStatus = Pending
  [ 117s] runtimeStatus = Running
  ⚠️ max_wait 1800s 초과 — 폴링 종료 (orchestration 은 계속 진행)

▶ 종료 (total=1805s)
⚠️ output 없음 또는 dict 아님:
{
  "name": "crawl_preprocess_orchestrator",
  "instanceId": "4ccedddd001e46afb5b733314a246f13",
  "runtimeStatus": "Running",
  "input": "{\"source\": \"all\", \"max_pages\": 1, \"detail_workers\": 5, \"triggered_by\": \"notebook-02\", \"skip_preprocess\": false, \"crawl_date\": \"2026-05-22\"}",
  "customStatus": null,
  "output": null,
  "createdTime": "2026-05-22T05:01:05Z",
  "lastUpdatedTime": "2026-05-22T05:20:11Z"
}


### 3-B. (옵션) Logic App 수동 트리거

위 셀 3 은 Durable orchestrator 를 **직접** 호출합니다.  
운영과 동일한 경로(Logic App → Function App)로 실행하려면 아래 셀을 사용하세요.

- Management API `POST .../triggers/{triggerName}/run` 으로 즉시 실행
- 결과는 **셀 4 (실행 이력)** 에서 모니터링

In [ ]:
import shutil, subprocess, json

def run_az(args):
    az_path = shutil.which("az") or "az"
    return subprocess.run([az_path] + args, capture_output=True, text=True)

sub_id = run_az(["account", "show", "--query", "id", "-o", "tsv"]).stdout.strip()

# Logic App 이름 자동 탐색
workflow_name = LOGIC_APP.strip() if LOGIC_APP else ""
if not workflow_name:
    res = run_az([
        "resource", "list",
        "--resource-group", RG_NAME,
        "--resource-type", "Microsoft.Logic/workflows",
        "--query", "[?contains(name,'logic-crawl')].name",
        "-o", "tsv",
    ])
    names = [n.strip() for n in res.stdout.splitlines() if n.strip()]
    if names:
        workflow_name = names[0]

assert workflow_name, "Logic App 을 찾을 수 없습니다."
trigger_name = "Daily_Schedule_Crawl_and_Preprocess"

print(f"▶ Logic App 수동 트리거")
print(f"  workflow : {workflow_name}")
print(f"  trigger  : {trigger_name}")

# 트리거 즉시 실행
run_uri = (
    f"https://management.azure.com/subscriptions/{sub_id}"
    f"/resourceGroups/{RG_NAME}/providers/Microsoft.Logic/workflows/{workflow_name}"
    f"/triggers/{trigger_name}/run?api-version=2019-05-01"
)
res = run_az(["rest", "--method", "post", "--uri", run_uri])
if res.returncode != 0:
    print(f"  ❌ 트리거 실패: {res.stderr.strip()[:300]}")
else:
    print(f"  ✅ 트리거 성공 (202 Accepted)")

# 최신 run 상태 확인
import time
time.sleep(3)
runs_uri = (
    f"https://management.azure.com/subscriptions/{sub_id}"
    f"/resourceGroups/{RG_NAME}/providers/Microsoft.Logic/workflows/{workflow_name}"
    f"/runs?$top=3&api-version=2019-05-01"
)
res = run_az(["rest", "--method", "get", "--uri", runs_uri])
if res.returncode == 0:
    runs = (json.loads(res.stdout) or {}).get("value", [])
    if runs:
        print(f"\n  최근 {len(runs)}건:")
        print(f"  {'startTime':<25} {'status':<12}")
        print("  " + "-" * 40)
        for r in runs:
            p = r.get("properties", {}) or {}
            st = p.get("startTime", "")[:19].replace("T", " ")
            print(f"  {st:<25} {p.get('status', '?'):<12}")
    print(f"\n  → 셀 4 (실행 이력) 에서 완료까지 모니터링하세요.")

## 4. (운영 모니터링) Logic App 정기 실행 이력

Logic App `Daily_Schedule_Crawl_and_Preprocess` 의 최근 20건 run 상태/실패 건수를 조회합니다. 위 셀 3 은 노트북 직접 호출 경로라 이 이력에는 잡히지 않습니다.


In [5]:
import shutil

def run_az(args):
    az_path = shutil.which("az") or "az"
    return subprocess.run([az_path] + args, capture_output=True, text=True)

sub_id = run_az(["account", "show", "--query", "id", "-o", "tsv"]).stdout.strip()

# Consumption Logic App (Microsoft.Logic/workflows) 자동 검색
workflow_name = LOGIC_APP.strip() if LOGIC_APP else ""
if not workflow_name:
    res = run_az([
        "resource", "list",
        "--resource-group", RG_NAME,
        "--resource-type", "Microsoft.Logic/workflows",
        "--query", "[?contains(name,'logic-crawl')].name",
        "-o", "tsv",
    ])
    names = [n.strip() for n in res.stdout.splitlines() if n.strip()]
    if names:
        workflow_name = names[0]

print(f"Logic workflow: {workflow_name}")

if not workflow_name:
    print("⚠️  Logic App 을 찾을 수 없습니다 — 이력 조회 생략")
else:
    api_ver = "2019-05-01"
    url = (
        f"https://management.azure.com/subscriptions/{sub_id}"
        f"/resourceGroups/{RG_NAME}/providers/Microsoft.Logic/workflows/{workflow_name}"
        f"/runs?api-version={api_ver}&$top=20"
    )
    res = run_az(["rest", "--method", "get", "--uri", url])
    if res.returncode != 0:
        print(f"실행 이력 조회 실패: {res.stderr.strip()[:300]}")
    else:
        runs = (json.loads(res.stdout) or {}).get("value", [])
        if not runs:
            print("  (실행 이력 없음 — 아직 트리거된 적 없음)")
        else:
            print(f"  최근 {len(runs)}건:")
            print(f"  {'startTime':<25} {'status':<12} {'duration_s':>10}  name")
            print("  " + "-" * 70)
            for r in runs:
                p = r.get("properties", {}) or {}
                st = p.get("startTime", "")[:19].replace("T", " ")
                et = p.get("endTime", "") or ""
                status = p.get("status", "?")
                dur = ""
                if st and et:
                    try:
                        s = datetime.fromisoformat(st)
                        e = datetime.fromisoformat(et[:19])
                        dur = f"{(e - s).total_seconds():.0f}"
                    except Exception:
                        dur = ""
                print(f"  {st:<25} {status:<12} {dur:>10}  {r.get('name','')}")


Logic workflow: logic-crawl-ragi-63325wdo
  최근 7건:
  startTime                 status       duration_s  name
  ----------------------------------------------------------------------
  2026-05-27 21:00:57       Failed              336  08584216912274881102202658141CU08
  2026-05-26 21:00:58       Failed              332  08584217776268595537623291486CU06
  2026-05-25 21:00:58       Failed              332  08584218640269354341978315728CU25
  2026-05-24 21:00:58       Failed              334  08584219504267430230850782813CU12
  2026-05-23 21:00:58       Failed              334  08584220368266392267950948262CU12
  2026-05-22 21:00:58       Failed              332  08584221232266923786645757980CU02
  2026-05-22 04:17:58       Failed                3  08584221834067324724065311471CU07


## 5. AI Search 인덱서 상태 확인

크롤링·전처리가 끝나면 AI Search 인덱서가 자동으로 새 문서를 색인합니다. 아래 셀로 마지막 실행 결과를 확인합니다.


In [6]:
# AI Search 인덱서 상태 확인
import os
from azure.identity import DefaultAzureCredential
import requests as req

SEARCH_ENDPOINT = os.environ.get('AZURE_SEARCH_SERVICE_ENDPOINT', '')
INDEXER_NAME = os.environ.get('AZURE_SEARCH_INDEXER_NAME', 'prec-court-indexer')

credential = DefaultAzureCredential()
token = credential.get_token("https://search.azure.com/.default").token

r = req.get(
    f"{SEARCH_ENDPOINT}/indexers/{INDEXER_NAME}/status?api-version=2024-07-01",
    headers={"Authorization": f"Bearer {token}"}
)
status = r.json()
last_run = status.get("lastResult", {})
print(f"인덱서: {INDEXER_NAME}")
print(f"인덱서 상태: {last_run.get('status', 'N/A')}")
print(f"처리된 문서: {last_run.get('itemsProcessed', 0)}")
print(f"실패한 문서: {last_run.get('itemsFailed', 0)}")
print(f"완료 시각: {last_run.get('endTime', 'N/A')}")

인덱서: prec-court-indexer
인덱서 상태: N/A
처리된 문서: 0
실패한 문서: 0
완료 시각: N/A
